# NOTEBOOK 1 - Data Preparation & Augmentation (`tongue-dataprep`)

This notebook:
1. Loads `t2d_tongue_dataset` from Kaggle Input.
2. Prints class distribution before augmentation.
3. Applies **offline** Albumentations augmentation per class (with controlled oversampling for balance).
4. Splits the **fully augmented** dataset into `70/15/15` (`train/val/test`) with stratification.
5. Prevents leakage by keeping each original image and all its augmentations in the same split.
6. Saves final folders directly to `/kaggle/working/{train,val,test}/class_name/`.

Publish `/kaggle/working` as a Kaggle Dataset named `tongue_preped`.

In [ ]:
# If needed on Kaggle, uncomment:
# !pip install -q albumentations==1.4.18 opencv-python-headless==4.10.0.84

import os
import cv2
import math
import shutil
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import albumentations as A
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_BASE = Path('/kaggle/input')
OUTPUT_BASE = Path('/kaggle/working')
SPLITS = ('train', 'val', 'test')
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

assert sum(SPLIT_RATIOS) == 1.0, 'Split ratios must sum to 1.0'

In [ ]:
def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS


def discover_dataset_root() -> Path:
    # Try direct expected names first
    preferred = [
        INPUT_BASE / 't2d_tongue_dataset',
        INPUT_BASE / 't2d-tongue-dataset',
    ]
    for p in preferred:
        if p.exists() and p.is_dir():
            return p

    # Fallback: find first directory under /kaggle/input that contains class folders with images
    for dataset_dir in INPUT_BASE.iterdir():
        if not dataset_dir.is_dir():
            continue
        subdirs = [d for d in dataset_dir.iterdir() if d.is_dir()]
        if not subdirs:
            continue
        has_images = any(any(is_image_file(f) for f in s.rglob('*')) for s in subdirs)
        if has_images:
            return dataset_dir

    raise FileNotFoundError('Could not find dataset root in /kaggle/input')


def collect_originals(root: Path):
    class_to_files = defaultdict(list)

    # Support either root/class/*.jpg OR deeper layouts (train/val/test/class/*.jpg)
    # We infer class as the parent directory name of each image.
    for img_path in root.rglob('*'):
        if img_path.is_file() and is_image_file(img_path):
            class_name = img_path.parent.name.strip().lower()
            class_to_files[class_name].append(img_path)

    # Keep only folders that have actual images
    class_to_files = {k: sorted(v) for k, v in class_to_files.items() if len(v) > 0}

    if len(class_to_files) < 2:
        raise ValueError(
            f'Expected at least 2 classes, found {len(class_to_files)}. '
            'Check your dataset folder layout.'
        )

    return class_to_files


def print_distribution(title: str, counts: dict):
    print(f'\n{title}')
    df = pd.DataFrame({'class': list(counts.keys()), 'count': list(counts.values())})
    print(df.sort_values('class').to_string(index=False))

In [ ]:
dataset_root = discover_dataset_root()
print(f'Dataset root: {dataset_root}')

class_to_originals = collect_originals(dataset_root)
original_counts = {cls: len(paths) for cls, paths in class_to_originals.items()}
print_distribution('Class distribution BEFORE augmentation:', original_counts)

In [ ]:
# Albumentations pipeline (prefer Albumentations over custom code)
augmenter = A.Compose([
    A.OneOf([
        A.HorizontalFlip(p=1.0),
        A.VerticalFlip(p=1.0),
        A.RandomRotate90(p=1.0),
    ], p=0.6),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=25,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.8,
    ),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
        A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=1.0),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=1.0),
    ], p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(std_range=(0.03, 0.08), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.35),
], p=1.0)


def augment_once(img_bgr: np.ndarray) -> np.ndarray:
    aug = augmenter(image=img_bgr)
    return aug['image']


# Build family records: each original image = one family
# We keep each family in exactly one split to prevent leakage.
families = []
for cls, paths in class_to_originals.items():
    for i, p in enumerate(paths):
        family_id = f'{cls}_{i:06d}'
        families.append({
            'family_id': family_id,
            'class_name': cls,
            'src_path': p,
        })

# Augmentation policy:
# - At least 1 augmentation per original for all classes
# - Then controlled oversampling to balance classes to the max class size after base augmentation
base_aug_per_image = 1
base_counts = {
    cls: len(paths) * (1 + base_aug_per_image)
    for cls, paths in class_to_originals.items()
}
target_count = max(base_counts.values())

# Plan number of augmentations per family
family_aug_counts = {f['family_id']: base_aug_per_image for f in families}

for cls, paths in class_to_originals.items():
    current = base_counts[cls]
    needed = target_count - current
    if needed <= 0:
        continue

    class_family_ids = [f['family_id'] for f in families if f['class_name'] == cls]

    # Distribute extra augmentations in round-robin fashion across class families
    idx = 0
    while needed > 0:
        fid = class_family_ids[idx % len(class_family_ids)]
        family_aug_counts[fid] += 1
        needed -= 1
        idx += 1

augmented_counts = defaultdict(int)
for f in families:
    cls = f['class_name']
    fid = f['family_id']
    augmented_counts[cls] += 1 + family_aug_counts[fid]

print_distribution('Class distribution AFTER augmentation (planned):', dict(augmented_counts))


# Stratified split by family labels (70/15/15)
family_ids = [f['family_id'] for f in families]
labels = [f['class_name'] for f in families]

train_ids, temp_ids, train_y, temp_y = train_test_split(
    family_ids,
    labels,
    test_size=(1 - SPLIT_RATIOS[0]),
    random_state=SEED,
    stratify=labels,
)

val_ratio_within_temp = SPLIT_RATIOS[1] / (SPLIT_RATIOS[1] + SPLIT_RATIOS[2])
val_ids, test_ids, _, _ = train_test_split(
    temp_ids,
    temp_y,
    test_size=(1 - val_ratio_within_temp),
    random_state=SEED,
    stratify=temp_y,
)

split_map = {}
for fid in train_ids:
    split_map[fid] = 'train'
for fid in val_ids:
    split_map[fid] = 'val'
for fid in test_ids:
    split_map[fid] = 'test'

# Sanity: every family appears once
assert len(split_map) == len(family_ids)

# Since each family always has >=1 augmented sample, each split will include augmented images
split_augmented_image_counts = defaultdict(int)
for f in families:
    fid = f['family_id']
    split_name = split_map[fid]
    split_augmented_image_counts[split_name] += family_aug_counts[fid]

print('\nAugmented images per split (planned):')
for s in SPLITS:
    print(f'  {s}: {split_augmented_image_counts[s]}')

In [ ]:
def safe_stem(name: str) -> str:
    # Keep filenames portable for Kaggle dataset publishing
    allowed = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_')
    return ''.join(ch if ch in allowed else '_' for ch in name)


# Prepare final output folders directly (no intermediate shared folder)
for split_name in SPLITS:
    split_dir = OUTPUT_BASE / split_name
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

for cls in class_to_originals.keys():
    for split_name in SPLITS:
        (OUTPUT_BASE / split_name / cls).mkdir(parents=True, exist_ok=True)


# Save originals + offline augmented images directly into split folders
final_counts = defaultdict(lambda: defaultdict(int))  # final_counts[split][class]

for f in tqdm(families, desc='Saving augmented dataset'):
    fid = f['family_id']
    cls = f['class_name']
    src = f['src_path']
    split_name = split_map[fid]

    img = cv2.imread(str(src))
    if img is None:
        print(f'[WARN] Could not read image: {src}')
        continue

    dst_dir = OUTPUT_BASE / split_name / cls

    # Save original
    orig_base = safe_stem(src.stem)
    orig_name = f'{fid}__orig__{orig_base}.jpg'
    cv2.imwrite(str(dst_dir / orig_name), img)
    final_counts[split_name][cls] += 1

    # Save offline augmentations planned for this family
    n_aug = family_aug_counts[fid]
    for j in range(n_aug):
        aug_img = augment_once(img)
        aug_name = f'{fid}__aug_{j:03d}__{orig_base}.jpg'
        cv2.imwrite(str(dst_dir / aug_name), aug_img)
        final_counts[split_name][cls] += 1


# Final report per class per split
rows = []
for split_name in SPLITS:
    for cls in sorted(class_to_originals.keys()):
        rows.append({
            'split': split_name,
            'class': cls,
            'count': final_counts[split_name][cls],
        })

final_df = pd.DataFrame(rows)
print('\nFinal image counts per class per split:')
print(final_df.to_string(index=False))

print('\nTotal images written:', int(final_df['count'].sum()))
print('Output root ready for Kaggle Dataset publish:', OUTPUT_BASE)

## Publish as Kaggle Dataset: `tongue_preped`

After running all cells:
1. Confirm the folders exist:
   - `/kaggle/working/train/<class_name>/`
   - `/kaggle/working/val/<class_name>/`
   - `/kaggle/working/test/<class_name>/`
2. In Kaggle Notebook right panel, click **Save Version**.
3. Choose **Save & Run All**.
4. In version options, enable **Create dataset from notebook output**.
5. Name dataset: `tongue_preped`.

This dataset can then be imported by your training notebook.